06 Error analysis, scene 3 (U-Net base 16, seed 0)
Tags: L label covers land with no visible fire, M visible fire inside the label that the model missed, U visible fire or fresh burn with no label, S smoke or cloud confusion, O other, ? cannot tell.

Run order: setup, sample, viewer. Then repeat: show_batch(N), then record(...). Tags and notes live in results/error_sample_scene3.csv, so restarting the kernel loses nothing.

In [ ]:
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

P = Path(r"G:\sen2fire-thesis")
EX = P / "data" / "sen2fire"
REFZIP = P / "data" / "reference" / "Baseline_code.zip"
SAMPLE_CSV = P / "results" / "error_sample_scene3.csv"
TAG_CODES = {"L", "M", "U", "S", "O", "?"}

d = np.load(P / "data" / "unet_b16_s3000_val_scores.npz")
SC = d["S"]                      # logits, float16, shape (504, 512, 512)
LB = d["YV"].astype(bool)        # labels
N_PATCH = len(SC)

# threshold at 1% false-positive rate over scene-3 pixels (same recipe as before)
_rng = np.random.default_rng(0)
_neg = SC[~LB]
THR1 = float(np.quantile(_neg[_rng.integers(0, _neg.size, 2_000_000)], 0.99))
del _neg
print("threshold (logit):", round(THR1, 4), "| earlier run: 1.5244")

LAB_PX = LB.reshape(N_PATCH, -1).sum(1)
DET_PX = (SC > THR1).reshape(N_PATCH, -1).sum(1)
HIT_PX = ((SC > THR1) & LB).reshape(N_PATCH, -1).sum(1)

with zipfile.ZipFile(REFZIP) as zf:
    VAL_ORDER = [l.strip().replace("\\", "/") for l in zf.read("dataset/val.txt").decode().splitlines() if l.strip()]

# check that row i of the score array is patch VAL_ORDER[i]
for i in np.random.default_rng(1).choice(N_PATCH, 12, replace=False):
    with np.load(EX / VAL_ORDER[i]) as z:
        assert np.array_equal(z["label"].astype(bool), LB[i]), f"order mismatch at {i}"
print("patch order check passed")


In [ ]:
# Sample: loads the saved CSV if it exists (keeps your tags); otherwise draws it with fixed seeds.
if SAMPLE_CSV.exists():
    sample = pd.read_csv(SAMPLE_CSV, dtype={"tag": str, "note": str}, keep_default_na=False)
    print("loaded", len(sample), "rows from", SAMPLE_CSV.name)
else:
    miss_c = np.where((LAB_PX >= 1000) & (HIT_PX < 0.05 * LAB_PX))[0]
    fa_c = np.where((LAB_PX == 0) & (DET_PX >= 500))[0]
    print("miss candidates", len(miss_c), "| false-alarm candidates", len(fa_c))
    _r = np.random.default_rng(123)
    pick = lambda c: _r.choice(c, size=min(20, len(c)), replace=False)
    rows = [(int(i), "miss") for i in pick(miss_c)] + [(int(i), "false alarm") for i in pick(fa_c)]
    rows = [rows[j] for j in _r.permutation(len(rows))]
    sample = pd.DataFrame(rows, columns=["idx", "kind"])
    sample["label_px"] = LAB_PX[sample["idx"].to_numpy()]
    sample["det_px"] = DET_PX[sample["idx"].to_numpy()]
    sample["tag"] = ""
    sample["note"] = ""
    sample.to_csv(SAMPLE_CSV, index=False)

print(sample["tag"].replace("", "(blank)").value_counts())


In [ ]:
def stretch(a):
    out = np.zeros_like(a)
    for c in range(3):
        lo, hi = np.percentile(a[..., c], (2, 98))
        out[..., c] = np.clip((a[..., c] - lo) / (hi - lo + 1e-6), 0, 1)
    return out


def show_batch(b, size=5):
    sub = sample.iloc[b * size:(b + 1) * size]
    fig, ax = plt.subplots(len(sub), 3, figsize=(12, 3.9 * len(sub)))
    ax = np.atleast_2d(ax)
    for r, (_, row) in enumerate(sub.iterrows()):
        i = int(row["idx"])
        with np.load(EX / VAL_ORDER[i]) as z:
            im = z["image"].astype(np.float32)
        rgb = stretch(np.stack([im[3], im[2], im[1]], -1))
        swir = stretch(np.stack([im[11], im[7], im[3]], -1))
        det = SC[i] > THR1
        for c, (bg, t) in enumerate([(rgb, "RGB"), (swir, "SWIR 11,7,3")]):
            ax[r, c].imshow(bg)
            if LB[i].any(): ax[r, c].contour(LB[i], levels=[0.5], colors="cyan", linewidths=1)
            if det.any(): ax[r, c].contour(det, levels=[0.5], colors="red", linewidths=1)
            ax[r, c].set_title(f"#{i} {row['kind']}: {t}", fontsize=8)
        ax[r, 2].imshow(SC[i], vmin=THR1 - 2, vmax=THR1 + 2, cmap="magma")
        ax[r, 2].set_title(f"score (mid = threshold) | label {row['label_px']} px | detected {row['det_px']} px", fontsize=8)
        for a in ax[r]: a.axis("off")
    plt.tight_layout(); plt.show()
    print("tag these patch indices:", [int(i) for i in sub["idx"]])


def record(tags=None, notes=None):
    ids = set(sample["idx"])
    for i, t in (tags or {}).items():
        assert i in ids, f"{i} is not in the sample"
        assert t in TAG_CODES, f"bad tag {t!r} for {i}"
        sample.loc[sample["idx"] == i, "tag"] = t
    for i, n in (notes or {}).items():
        assert i in ids, f"{i} is not in the sample"
        sample.loc[sample["idx"] == i, "note"] = n
    sample.to_csv(SAMPLE_CSV, index=False)
    print(sample["tag"].replace("", "(blank)").value_counts())


Batch 2 (36 patches in 8 batches, numbered 0 to 7)
View the batch, then fill the dictionaries in the cell below it.

In [ ]:
show_batch(2)

In [ ]:
record(
    tags={},      # e.g. {123: "L", 45: "M"}
    notes={},     # e.g. {123: "corner box over haze"}
)

In [ ]:
print(pd.crosstab(sample["kind"], sample["tag"].replace("", "(blank)"), margins=True))
sample[sample["note"] != ""][["idx", "kind", "tag", "note"]]

In [ ]:
record(
    tags={97: "U", 325: "U", 86: "L", 12: "U", 179: "?"},
    notes={325: "3 hot spots; one just below label at right edge, others unlabelled",
           179: "label under thick cloud/smoke, cannot verify"},
)

In [ ]:
show_batch(3)

In [ ]:
record(
    tags={62: "U", 176: "L", 96: "L", 286: "U", 327: "?"},
    notes={62: "small hot spot at patch corner",
           286: "small hot spot at patch corner",
           327: "label under thick cloud, cannot verify"},
)

In [ ]:
show_batch(4)

In [ ]:
def show_one(i):
    with np.load(EX / VAL_ORDER[i]) as z:
        im = z["image"].astype(np.float32)
    fig, ax = plt.subplots(1, 3, figsize=(18, 6))
    ax[0].imshow(stretch(np.stack([im[3], im[2], im[1]], -1)))
    ax[1].imshow(stretch(np.stack([im[11], im[7], im[3]], -1)))
    ax[2].imshow(im[11], cmap="inferno", vmin=np.percentile(im[11], 2), vmax=np.percentile(im[11], 99.5))
    for a, t in zip(ax, ["RGB", "SWIR 11,7,3", "channel 11 (SWIR)"]):
        if LB[i].any(): a.contour(LB[i], levels=[0.5], colors="cyan", linewidths=1)
        a.set_title(f"#{i} {t}", fontsize=9); a.axis("off")
    plt.tight_layout(); plt.show()

show_one(201)

In [ ]:
record(
    tags={319: "L", 47: "U", 201: "L", 291: "S", 27: "U"},
    notes={201: "dark block in label, looks like dark land not fire; checked at zoom",
           291: "detections on smoke edge top-left; label under smoke"},
)

In [ ]:
show_batch(5)

In [ ]:
record(
    tags={287: "U", 356: "L", 355: "L", 103: "L", 357: "L", 325: "L"},
    notes={356: "pink patch at right end of box; low confidence",
           357: "separate hot core at bottom, outside label (unlabelled fire)"},
)

In [ ]:
show_batch(6)

In [ ]:
record(
    tags={329: "L", 7: "L", 323: "U", 133: "U", 359: "U"},
    notes={133: "hot spot at top edge of patch"},
)

In [ ]:
show_batch(7)

In [ ]:
record(tags={68: "U"})